# Import

In [1]:
from pathlib import Path
import requests
import gzip
import xml.etree.ElementTree as ET
import json
import time
from tqdm.notebook import tqdm

In [2]:
# Dictionnaire associant le nom du média (utilisé comme identifiant de fichier) 
# à l'URL de son sitemap d'index ou à une liste de sous-sitemaps explicites.
media = {
    "le_figaro": "https://sitemaps.lefigaro.fr/lefigaro.fr/articles.xml",
    "le_capital": "https://www.capital.fr/sitemap/articles.xml",
    "nice_matin": "https://www.nicematin.com/sitemap.xml",
    "le_nouvel_observateur": "https://www.nouvelobs.com/sitemap/sitemap-index-articles.xml",
    "le_journal_du_dimanche": "https://www.lejdd.fr/sitemap.xml",
    "le_monde": "https://www.lemonde.fr/sitemap_index.xml",
    "les_echos": "https://sitemap.lesechos.fr/sitemap_index.xml",
    "telerama": "https://www.telerama.fr/sitemaps/sitemap_index.php",
    "valeurs_actuelles": [
        "https://www.valeursactuelles.com/post-sitemap.xml",
        "https://www.valeursactuelles.com/post-sitemap2.xml",
        "https://www.valeursactuelles.com/post-sitemap3.xml",
    ],
}

# Liste d'URLs de sitemaps à ignorer (ex: sitemaps de catégories ou ressources non pertinentes).
to_ignore = {
    "https://www.nicematin.com/sitemap_fixed.xml",
    "https://www.nouvelobs.com/sitemap/sitemap-index-categories.xml",
}


In [ ]:
# Répertoire de destination pour la sauvegarde des données extraites.
OUTPUT_DIR = Path("C:/Users/E.E/Desktop/STAGE/Dashboard/stage-mids/data/processed")

# Télécharge et retourne le contenu d'un fichier XML. Gère la décompression automatique si le format est GZIP.
def read_xml(url_cible):
    raw_content = requests.get(url_cible, timeout=60).content
    
    if not raw_content:
        raise ValueError("Contenu vide retourné par la requête.")

    # Décompression si le contenu est au format GZIP    
    if raw_content[:2] == b'\x1f\x8b':
        raw_content = gzip.decompress(raw_content)
        
    return raw_content


for media_name, sitemap in tqdm(media.items(), desc="Médias", unit="média"):

    # Création du fichier de sortie pour le média courant
    DATA_FILE = OUTPUT_DIR / f"{media_name}_articles.jsonl"
    
    # Initialisation avec les sitemaps à ignorer
    checked_sitemap = set(to_ignore)

    # Chargement de l'historique des traitements si le fichier de sortie existe déjà.
    if DATA_FILE.exists():
        with open(DATA_FILE, "r", encoding="utf-8") as f:
            for line in f:
                checked_sitemap.add(json.loads(line)["sitemap"])

    # Si la source est une liste, on l'utilise directement comme liste de sous-sitemaps.
    if isinstance(sitemap, list):
        sub_sitemaps = sitemap
    
    # Sinon, on télécharge le sitemap d'index pour en extraire la liste des sous-sitemaps.
    else:
        raw_xml = read_xml(sitemap)
        root = ET.fromstring(raw_xml)
        sub_sitemaps = [loc.text for loc in root.findall(".//{http://www.sitemaps.org/schemas/sitemap/0.9}loc")]

    for sub_sitemap in tqdm(sub_sitemaps, desc=f"{media_name}", unit="sitemap", leave=False):
        
        if sub_sitemap in checked_sitemap:
            continue

        try:
            raw_xml = read_xml(sub_sitemap)
            root = ET.fromstring(raw_xml)
            urls = [loc.text for loc in root.findall(".//{http://www.sitemaps.org/schemas/sitemap/0.9}loc")]
            result = {"type": "ok", "sitemap": sub_sitemap, "urls": urls}
        except Exception:
            result = {"type": "echec", "sitemap": sub_sitemap}

        # Sauvegarde et marquage en une seule étape
        with open(DATA_FILE, "a", encoding="utf-8") as f:
            f.write(json.dumps(result) + "\n")
        checked_sitemap.add(sub_sitemap)

        time.sleep(1)